In [1]:
#import libraries
import psycopg2
import numpy as np
import re
import dash
from dash import dcc, html, Input, Output, dash_table, State
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from dash.exceptions import PreventUpdate
from datetime import datetime
from dash.exceptions import PreventUpdate

In [2]:
#Reading data from postgresql

# Establish connection
conn = psycopg2.connect(
    host="localhost",
    database="APDV",
    user="dap",
    password="dap"
)

# Define query
sql_query = "SELECT * FROM airquality"

# Execute with parameters
params = ('value',)
airqualitydata = pd.read_sql(sql_query, conn, params=params)

# Display results
print(airqualitydata.head())

# Close connection
conn.close()

   index  state_code  county_code  site_number  parameter_code  poc parameter  \
0      0           1            3           10           44201    1     Ozone   
1      1           1            3           10           44201    1     Ozone   
2      2           1           49         9991           44201    1     Ozone   
3      3           1           51            4           44201    1     Ozone   
4      4           1           51            4           44201    1     Ozone   

    si_id  method_code                                method  ... state_name  \
0       7           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
1       7           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
2   96297           47             INSTRUMENTAL-ULTRA VIOLET  ...    Alabama   
3  104232           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
4  104232           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   

   county_name   city_name cbsa_

C:\Users\harig\AppData\Local\Temp\ipykernel_14180\261123222.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  airqualitydata = pd.read_sql(sql_query, conn, params=params)


In [3]:
airqualitydata.shape

(3751, 59)

In [4]:
#information
airqualitydata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3751 entries, 0 to 3750
Data columns (total 59 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   index                           3751 non-null   int64  
 1   state_code                      3751 non-null   int64  
 2   county_code                     3751 non-null   int64  
 3   site_number                     3751 non-null   int64  
 4   parameter_code                  3751 non-null   int64  
 5   poc                             3751 non-null   int64  
 6   parameter                       3751 non-null   object 
 7   si_id                           3751 non-null   int64  
 8   method_code                     3751 non-null   int64  
 9   method                          3751 non-null   object 
 10  assessment_date                 3751 non-null   object 
 11  assessment_number               3751 non-null   float64
 12  unit_code                       37

In [5]:
#descriptive statistics
airqualitydata.describe()

,index,state_code,county_code,site_number,parameter_code,poc,si_id,method_code,assessment_number,unit_code,...,lvl9_assessment_concentration,lvl10_monitor_concentration,lvl10_assessment_concentration,pqao_code,monitoring_agency_code,latitude,longitude,cbsa_code,csa_code,tribal_code
count,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,...,179.000000,107.000000,107.000000,3751.000000,3751.000000,3751.000000,3751.000000,3436.000000,2636.000000,64.000000
mean,1875.000000,28.826713,76.028792,1021.309251,43370.389496,1.183951,48186.347907,165.008531,1.026126,7.553452,...,151.340892,193.063675,193.035850,778.279126,755.504665,38.515080,-91.414872,30913.643772,343.529970,575.109375
std,1082.964758,15.281286,83.145099,2223.356373,891.625943,0.616890,44836.646849,180.511962,0.225944,0.497201,...,170.799230,118.816449,120.313015,370.451523,339.766735,4.662137,15.851393,10964.979616,126.243761,331.901932
min,0.000000,1.000000,1.000000,1.000000,42101.000000,1.000000,7.000000,9.000000,1.000000,7.000000,...,0.163200,0.210000,0.212000,13.000000,12.000000,19.060655,-159.366240,10220.000000,104.000000,17.000000
25%,937.500000,17.000000,21.000000,8.000000,42401.000000,1.000000,7109.000000,74.000000,1.000000,7.000000,...,0.179885,161.900000,165.100000,584.000000,584.000000,35.494530,-103.273777,19780.000000,216.000000,424.500000
50%,1875.000000,30.000000,59.000000,40.000000,44201.000000,1.000000,15413.000000,87.000000,1.000000,8.000000,...,0.326000,249.000000,249.000000,768.000000,768.000000,39.464872,-87.577222,33460.000000,368.000000,750.000000
75%,2812.500000,40.000000,103.000000,1006.000000,44201.000000,1.000000,96335.500000,100.000000,1.000000,8.000000,...,319.350000,253.000000,250.000000,1035.000000,1001.000000,41.591234,-78.771530,39830.000000,430.000000,788.750000
max,3750.000000,56.000000,800.000000,9997.000000,44201.000000,9.000000,105404.000000,600.000000,4.000000,8.000000,...,450.100000,415.000000,416.000000,6532.000000,6532.000000,64.845690,-67.061325,49740.000000,566.000000,920.000000


# Missing Data Handling

In [6]:
#percentage of missing value
airqualitydata.isnull().sum()/airqualitydata.shape[0]*100

index                               0.000000
state_code                          0.000000
county_code                         0.000000
site_number                         0.000000
parameter_code                      0.000000
poc                                 0.000000
parameter                           0.000000
si_id                               0.000000
method_code                         0.000000
method                              0.000000
assessment_date                     0.000000
assessment_number                   0.000000
unit_code                           0.000000
unit                                0.000000
lvl1_monitor_concentration         72.967209
lvl1_assessment_concentration      72.913890
lvl2_monitor_concentration         30.445215
lvl2_assessment_concentration      30.445215
lvl3_monitor_concentration         48.840309
lvl3_assessment_concentration      48.840309
lvl4_monitor_concentration         34.657425
lvl4_assessment_concentration      34.684084
lvl5_monit

In [7]:
def handle_missing_values(airqualitydata):
    """
    Handle missing values in a DataFrame with proper type checking
    """
    airqualitydata_clean = airqualitydata.copy()
    
    # 1. Drop high-missing columns
    cols_to_drop = [
        'auditing_agency_code', 'auditing_agency',
        'tribal_code', 'tribe_name',
        'lvl10_monitor_concentration', 'lvl10_assessment_concentration',
        'lvl9_monitor_concentration', 'lvl9_assessment_concentration'
    ]
    cols_to_drop = [col for col in cols_to_drop if col in airqualitydata_clean.columns]
    airqualitydata_clean = airqualitydata_clean.drop(columns=cols_to_drop)
    
    # 2. Process each column with type-specific handling
    for col in airqualitydata_clean.columns:
        if airqualitydata_clean[col].isna().sum() > 0:  # Only process columns with missing values
            try:
                # For numeric columns
                if pd.api.types.is_numeric_dtype(airqualitydata_clean[col]):
                    # Create missing indicator for concentration columns
                    if 'concentration' in col:
                        airqualitydata_clean[f'{col}_missing'] = airqualitydata_clean[col].isna().astype(int)
                    # Fill with median for numeric
                    median_val = airqualitydata_clean[col].median()
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(median_val)
                
                # For string/object columns
                elif pd.api.types.is_string_dtype(airqualitydata_clean[col]):
                    mode_val = airqualitydata_clean[col].mode()[0] if not airqualitydata_clean[col].mode().empty else 'Unknown'
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(mode_val)
                
                # For datetime columns
                elif pd.api.types.is_datetime64_any_dtype(airqualitydata_clean[col]):
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(method='ffill')
                
            except Exception as e:
                print(f"Could not process column {col}: {str(e)}")
                # Fallback to simple fill for problematic columns
                airqualitydata_clean[col] = airqualitydata_clean[col].fillna('Unknown') if pd.api.types.is_string_dtype(airqualitydata_clean[col]) else airqualitydata_clean[col].fillna(0)
    
    return airqualitydata_clean

In [8]:
# Process missing values
airqualitydata_clean = handle_missing_values(airqualitydata)



In [9]:
#percentage of missing value
airqualitydata_clean.isnull().sum()/airqualitydata_clean.shape[0]*100

index                                    0.0
state_code                               0.0
county_code                              0.0
site_number                              0.0
parameter_code                           0.0
                                        ... 
lvl6_assessment_concentration_missing    0.0
lvl7_monitor_concentration_missing       0.0
lvl7_assessment_concentration_missing    0.0
lvl8_monitor_concentration_missing       0.0
lvl8_assessment_concentration_missing    0.0
Length: 67, dtype: float64

# Transformantion

In [10]:
airqualitydata_clean.head()



,index,state_code,county_code,site_number,parameter_code,poc,parameter,si_id,method_code,method,...,lvl4_monitor_concentration_missing,lvl4_assessment_concentration_missing,lvl5_monitor_concentration_missing,lvl5_assessment_concentration_missing,lvl6_monitor_concentration_missing,lvl6_assessment_concentration_missing,lvl7_monitor_concentration_missing,lvl7_assessment_concentration_missing,lvl8_monitor_concentration_missing,lvl8_assessment_concentration_missing
0,0,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
1,1,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
2,2,1,49,9991,44201,1,Ozone,96297,47,INSTRUMENTAL-ULTRA VIOLET,...,0,0,1,1,0,0,1,1,1,1
3,3,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
4,4,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0


In [11]:
#Finding unique values in method column
airqualitydata_clean.method.value_counts()

method
INSTRUMENTAL-ULTRA VIOLET ABSORPTION                                                                                   1239
INSTRUMENTAL-ULTRA VIOLET                                                                                               713
INSTRUMENTAL-GAS PHASE CHEMILUMINESCENCE                                                                                267
INSTRUMENTAL-ULTRAVIOLET FLUORESCENCE                                                                                   228
INSTRUMENTAL-PULSED FLUORESCENT                                                                                         166
INSTRUMENTAL-Pulsed Fluorescent 43C-TLE/43i-TLE                                                                         153
Teledyne Model T500U-Cavity Attenuated Phase Shift Spectroscopy                                                         133
INSTRUMENTAL-CHEMILUMINESCENCE                                                                                          125
I

In [12]:
#Clean Method Names
airqualitydata_clean['method_clean'] = (
    airqualitydata_clean['method']
    .str.replace(
        r'(?i)(instrumental|analyzer)[\s\-]*(ultra[\s\-]*violet|uv)', 
        'UV', 
        regex=True
    )
    .str.replace(r'(?i)\b(absorption|spectrometry|analysis)\b', '', regex=True)
    .str.strip()
    .replace('', 'UV Method')
)

In [13]:
airqualitydata_clean.head()

,index,state_code,county_code,site_number,parameter_code,poc,parameter,si_id,method_code,method,...,lvl4_assessment_concentration_missing,lvl5_monitor_concentration_missing,lvl5_assessment_concentration_missing,lvl6_monitor_concentration_missing,lvl6_assessment_concentration_missing,lvl7_monitor_concentration_missing,lvl7_assessment_concentration_missing,lvl8_monitor_concentration_missing,lvl8_assessment_concentration_missing,method_clean
0,0,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,1,1,1,1,0,0,UV
1,1,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,1,1,1,1,0,0,UV
2,2,1,49,9991,44201,1,Ozone,96297,47,INSTRUMENTAL-ULTRA VIOLET,...,0,1,1,0,0,1,1,1,1,UV
3,3,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,1,1,1,1,0,0,UV
4,4,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,1,1,1,1,0,0,UV


In [32]:
airqualitydata_clean.columns

Index(['index', 'state_code', 'county_code', 'site_number', 'parameter_code',
       'poc', 'parameter', 'si_id', 'method_code', 'method', 'assessment_date',
       'assessment_number', 'unit_code', 'unit', 'lvl1_monitor_concentration',
       'lvl1_assessment_concentration', 'lvl2_monitor_concentration',
       'lvl2_assessment_concentration', 'lvl3_monitor_concentration',
       'lvl3_assessment_concentration', 'lvl4_monitor_concentration',
       'lvl4_assessment_concentration', 'lvl5_monitor_concentration',
       'lvl5_assessment_concentration', 'lvl6_monitor_concentration',
       'lvl6_assessment_concentration', 'lvl7_monitor_concentration',
       'lvl7_assessment_concentration', 'lvl8_monitor_concentration',
       'lvl8_assessment_concentration', 'pqao_code', 'pqao',
       'monitoring_agency_code', 'monitoring_agency', 'performing_agency_code',
       'performing_agency', 'analyzing_agency_code', 'analyzing_agency',
       'latitude', 'longitude', 'datum', 'local_site_name',

In [14]:
#Create Composite Site ID
airqualitydata_clean['site_id'] = (
    airqualitydata_clean['state_code'].astype(str).str.zfill(2) + '-' +
    airqualitydata_clean['county_code'].astype(str).str.zfill(3) + '-' +
    airqualitydata_clean['site_number'].astype(str).str.zfill(4)
)

In [15]:
# Extract Measurement Precision from Method
airqualitydata_clean['precision_code'] = (
    airqualitydata_clean['method']
    .str.extract(r'(\bCLASS\s*[IVX]+|\bPRECISION\s*\d+)', flags=re.IGNORECASE)
    .squeeze()
)

In [16]:
#Flag Incomplete Monitoring Data
missing_cols = [col for col in airqualitydata_clean.columns if 'missing' in col]
airqualitydata_clean['data_completeness'] = 1 - (airqualitydata_clean[missing_cols].sum(axis=1) / len(missing_cols))

In [17]:
#Categorize by Data Quality
conditions = [
    airqualitydata_clean['data_completeness'] >= 0.9,
    airqualitydata_clean['data_completeness'] >= 0.5,
]
choices = ['High', 'Medium']
airqualitydata_clean['quality_category'] = np.select(conditions, choices, default='Low')

In [18]:
#Parse Concentration Ranges
airqualitydata_clean['lvl1_monitor_concentration_clean'] = (
    airqualitydata_clean['lvl1_monitor_concentration']
    .astype(str)
    .str.extract(r'([<>]?)(\d+\.?\d*)')
    .apply(lambda x: 
        float(x[1]) * (-1 if x[0] == '<' else 1 if x[0] == '>' else 1), 
        axis=1
    )
)

In [19]:
airqualitydata_clean['has_lvl8_data'] = airqualitydata_clean['lvl8_monitor_concentration_missing'].eq(0).astype(int)

In [20]:
airqualitydata_clean['concentration_trend'] = (
    airqualitydata_clean.groupby('site_id')['lvl1_monitor_concentration']
    .transform(lambda x: x.diff().rolling(3, min_periods=1).mean())
)

In [21]:
airqualitydata_clean['agency_type'] = (
    airqualitydata_clean['monitoring_agency_code']
    .astype(str)
    .str.extract(r'^([A-Z]+)')[0]
    .replace({'EPA': 'Federal', 'STATE': 'State', None: 'Local'})
)

In [22]:
airqualitydata_clean['display_text'] = (
    "Site " + airqualitydata_clean['site_number'].astype(str) + 
    " (" + airqualitydata_clean['parameter'] + "): " + 
    airqualitydata_clean['lvl1_monitor_concentration'].round(2).astype(str) + " " + 
    airqualitydata_clean['unit'].fillna('ppm')
)

In [23]:
# Target encoding for counties
county_means = airqualitydata_clean.groupby('county_code')['lvl1_monitor_concentration'].mean().to_dict()
airqualitydata_clean['county_encoded'] = airqualitydata_clean['county_code'].map(county_means)

# Frequency encoding for parameters
param_freq = airqualitydata_clean['parameter'].value_counts(normalize=True).to_dict()
airqualitydata_clean['param_freq_encoded'] = airqualitydata_clean['parameter'].map(param_freq)

In [24]:
# Convert to datetime if not already
airqualitydata_clean['date_of_last_change'] = pd.to_datetime(airqualitydata_clean['date_of_last_change'])

# Extract temporal components
airqualitydata_clean['change_year'] = airqualitydata_clean['date_of_last_change'].dt.year
airqualitydata_clean['change_quarter'] = airqualitydata_clean['date_of_last_change'].dt.quarter
airqualitydata_clean['days_since_change'] = (pd.Timestamp.now() - airqualitydata_clean['date_of_last_change']).dt.days

In [25]:
# Cyclical encoding for seasons
airqualitydata_clean['change_month_sin'] = np.sin(2*np.pi*airqualitydata_clean['date_of_last_change'].dt.month/12)
airqualitydata_clean['change_month_cos'] = np.cos(2*np.pi*airqualitydata_clean['date_of_last_change'].dt.month/12)

In [26]:
# Method complexity (word count)
airqualitydata_clean['method_complexity'] = airqualitydata_clean['method'].str.split().str.len()

# Equipment type from method text
airqualitydata_clean['equipment_type'] = np.where(
    airqualitydata_clean['method'].str.contains('INSTRUMENTAL', case=False), 
    'Analytical', 
    'Manual'
)

In [27]:

# Season-concentration interaction
airqualitydata_clean['summer_highs'] = (airqualitydata_clean['date_of_last_change'].dt.month.isin([6,7,8])) & (airqualitydata_clean['lvl1_monitor_concentration'] > 60)

In [28]:
# Previous day's concentration
airqualitydata_clean['prev_day_concentration'] = airqualitydata_clean.groupby('site_number')['lvl1_monitor_concentration'].shift(1)

In [29]:
# Site-level percentiles
airqualitydata_clean['site_percentile'] = airqualitydata_clean.groupby('site_number')['lvl1_monitor_concentration'].rank(pct=True)

# County-level volatility
airqualitydata_clean['county_volatility'] = airqualitydata_clean.groupby(['county_code','change_year'])['lvl1_monitor_concentration'].transform('std')

In [30]:
airqualitydata_clean.shape

(3751, 90)

In [ ]:

# Data preprocessing
numeric_cols = ['lvl1_monitor_concentration', 'county_volatility', 'site_percentile']
airqualitydata_clean[numeric_cols] = airqualitydata_clean[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
airqualitydata_clean['date'] = pd.to_datetime(airqualitydata_clean['date_of_last_change'])  # Ensure datetime format

# Initialize Dash app
app = dash.Dash(__name__)
server = app.server

app.layout = html.Div([
    # Title and description
    html.Div([
        html.H1("Interactive Air Quality Dashboard", style={'textAlign': 'center'}),
        html.P("Explore air quality metrics across different locations and time periods", 
              style={'textAlign': 'center', 'color': '#7FDBFF'})
    ], style={'marginBottom': '30px'}),
    
    # Filters Row
    html.Div([
        # State Filter
        html.Div([
            html.Label("Select States:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='state-filter',
                options=[{'label': f"State {s}", 'value': s} for s in sorted(airqualitydata_clean['state_code'].unique())],
                multi=True,
                placeholder='All States',
                style={'width': '100%'}
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Parameter Filter
        html.Div([
            html.Label("Select Parameter:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='parameter-filter',
                options=[{'label': p, 'value': p} for p in airqualitydata_clean['parameter'].unique()],
                value='Ozone',
                style={'width': '100%'}
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Date Range Filter
        html.Div([
            html.Label("Date Range:", style={'fontWeight': 'bold'}),
            dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=airqualitydata_clean['date'].min(),
                max_date_allowed=airqualitydata_clean['date'].max(),
                start_date=airqualitydata_clean['date'].min(),
                end_date=airqualitydata_clean['date'].max()
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Concentration Threshold
        html.Div([
            html.Label("Concentration Threshold:", style={'fontWeight': 'bold'}),
            dcc.Slider(
                id='concentration-slider',
                min=0,
                max=airqualitydata_clean['lvl1_monitor_concentration'].max(),
                value=airqualitydata_clean['lvl1_monitor_concentration'].median(),
                marks={i: str(i) for i in range(0, int(airqualitydata_clean['lvl1_monitor_concentration'].max())+1, 10)}
            )
        ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'})
    ], style={'margin': '20px 0', 'backgroundColor': '#f8f9fa', 'borderRadius': '10px', 'padding': '15px'}),
    
    # Main Visualizations
    html.Div([
        # Map and Time Series
        html.Div([
            dcc.Graph(id='geo-map', style={'height': '500px'}),
            dcc.Graph(id='time-trend', style={'height': '500px'})
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Statistical Plots
        html.Div([
            dcc.Graph(id='violin-plot'),
            dcc.Graph(id='heatmap-plot')
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Additional Interactive Plots
        html.Div([
            dcc.Graph(id='animated-scatter'),
            dcc.Graph(id='parallel-coords')
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Equipment and Site Analysis
        html.Div([
            dcc.Graph(id='equipment-chart'),
            dcc.Graph(id='site-analysis')
        ], style={'display': 'flex', 'flexDirection': 'row'})
    ]),
    
    # Data Table and Export
    html.Div([
        html.H3("Filtered Data", style={'marginTop': '30px'}),
        html.Div([
            html.Button("Export to CSV", id='export-button', n_clicks=0),
            dcc.Download(id="download-dataframe-csv")
        ], style={'margin': '10px 0'}),
        dash_table.DataTable(
            id='data-table',
            columns=[{"name": i, "id": i} for i in airqualitydata_clean.columns],
            page_size=10,
            style_table={'overflowX': 'auto'},
            style_cell={'textAlign': 'left', 'padding': '10px'},
            style_header={'backgroundColor': '#2c3e50', 'color': 'white'},
            filter_action="native",
            sort_action="native"
        )
    ], style={'margin': '40px 0', 'padding': '15px', 'backgroundColor': '#f8f9fa', 'borderRadius': '10px'}),
    
    # Hidden div for storing filtered data
    html.Div(id='filtered-data-store', style={'display': 'none'})
])

# Callback to update all visualizations
@app.callback(
    [Output('geo-map', 'figure'),
     Output('time-trend', 'figure'),
     Output('violin-plot', 'figure'),
     Output('heatmap-plot', 'figure'),
     Output('animated-scatter', 'figure'),
     Output('parallel-coords', 'figure'),
     Output('equipment-chart', 'figure'),
     Output('site-analysis', 'figure'),
     Output('data-table', 'data'),
     Output('filtered-data-store', 'children')],
    [Input('state-filter', 'value'),
     Input('parameter-filter', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date'),
     Input('concentration-slider', 'value')]
)
def update_dashboard(selected_states, selected_param, start_date, end_date, concentration_threshold):
    # Filter data based on selections
    filtered_df = airqualitydata_clean.copy()
    
    if selected_param:
        filtered_df = filtered_df[filtered_df['parameter'] == selected_param]
    
    if selected_states:
        filtered_df = filtered_df[filtered_df['state_code'].isin(selected_states)]
    
    filtered_df = filtered_df[
        (filtered_df['date'] >= start_date) & 
        (filtered_df['date'] <= end_date) &
        (filtered_df['lvl1_monitor_concentration'] >= concentration_threshold)
    ]
    
    if filtered_df.empty:
        raise PreventUpdate
    
    # 1. Interactive Geospatial Map with click events
    geo_fig = px.scatter_mapbox(
        filtered_df,
        lat='latitude',
        lon='longitude',
        color='lvl1_monitor_concentration',
        size='county_volatility',
        hover_name='county_name',
        hover_data=['site_number', 'parameter', 'date'],
        color_continuous_scale=px.colors.sequential.Plasma,
        zoom=5,
        title='Interactive Air Quality Map (click on points for details)',
        height=500
    )
    geo_fig.update_layout(
        mapbox_style="open-street-map",
        clickmode='event+select'
    )
    
    # 2. Time-Series with Range Selector
    trend_fig = px.line(
        filtered_df.groupby(['date', 'change_quarter'])['lvl1_monitor_concentration'].mean().reset_index(),
        x='date',
        y='lvl1_monitor_concentration',
        color='change_quarter',
        title='Concentration Trends with Range Selector',
        height=500
    )
    trend_fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    )
    
    # 3. Interactive Violin Plot with Box Plot
    violin_fig = px.violin(
        filtered_df,
        y='lvl1_monitor_concentration',
        x='equipment_type',
        box=True,
        points="all",
        hover_data=['site_number', 'county_name'],
        title='Distribution by Equipment Type (hover for details)'
    )
    
    # 4. Heatmap of Concentration by Time and Location
    heatmap_fig = px.density_heatmap(
        filtered_df,
        x='date',
        y='county_name',
        z='lvl1_monitor_concentration',
        title='Concentration Heatmap by Date and County',
        height=400
    )
    
    # 5. Animated Scatter Plot
    animated_fig = px.scatter(
        filtered_df,
        x='change_year',
        y='lvl1_monitor_concentration',
        size='county_volatility',
        color='equipment_type',
        animation_frame='change_quarter',
        hover_name='county_name',
        title='Seasonal Changes in Air Quality (Animated)',
        height=400
    )
    
    # 6. Parallel Coordinates Plot
    parallel_fig = px.parallel_coordinates(
        filtered_df,
        dimensions=['lvl1_monitor_concentration', 'county_volatility', 'site_percentile', 'method_complexity'],
        color='lvl1_monitor_concentration',
        title='Multidimensional Analysis',
        height=400
    )
    
    # 7. Interactive Sunburst Chart
    equip_fig = px.sunburst(
        filtered_df,
        path=['state_code', 'county_name', 'equipment_type'],
        values='site_number',
        color='lvl1_monitor_concentration',
        title='Equipment Distribution Hierarchy',
        height=400
    )
    
    # 8. Site Analysis Scatter Plot
    site_fig = px.scatter(
        filtered_df,
        x='days_since_change',
        y='lvl1_monitor_concentration',
        color='county_name',
        size='county_volatility',
        hover_data=['site_number', 'method'],
        title='Site Performance Analysis',
        height=400
    )
    
    # Data Table
    table_data = filtered_df.to_dict('records')
    
    # Store filtered data for export
    stored_data = filtered_df.to_json(date_format='iso', orient='split')
    
    return geo_fig, trend_fig, violin_fig, heatmap_fig, animated_fig, parallel_fig, equip_fig, site_fig, table_data, stored_data

# Callback for exporting data
@app.callback(
    Output("download-dataframe-csv", "data"),
    Input("export-button", "n_clicks"),
    State('filtered-data-store', 'children'),
    prevent_initial_call=True
)
def export_data(n_clicks, stored_data):
    if n_clicks > 0 and stored_data:
        filtered_df = pd.read_json(stored_data, orient='split')
        return dcc.send_data_frame(filtered_df.to_csv, "filtered_air_quality_data.csv")
    raise PreventUpdate

# Run the app
if __name__ == '__main__':
    #app.run(jupyter_mode='inline', debug=True)  
    app.run(jupyter_mode='external', port=8053)# debug=True helps see errors

Dash app running on http://127.0.0.1:8053/


C:\Users\harig\AppData\Local\Temp\ipykernel_14180\1237135989.py:156: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\harig\AppData\Local\Temp\ipykernel_14180\1237135989.py:156: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\harig\AppData\Local\Temp\ipykernel_14180\1237135989.py:156: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\harig\AppData\Local\Temp\ipykernel_14180\1237135989.py:156: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\harig\AppData\Local\Temp\ipykernel_14180\1237135989.py:156: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn m